# HPDM097 Assignment 2
## Iteration 1: Arrivals (Gemini)
In this first attempt, the LLM was used to attempt to model the stroke patient arrivals process using SimPy. The LLM used was Gemini 3 Pro

Prompt and response: https://gemini.google.com/share/551c38e3e8f3

## Prompt:
Please create some python code to simulate the arrival of patients and flow through an acute and community stroke service. This will be a discrete event simulation built using an iterative process. We will start with the code to simulate arrivals, and work on the flow through the system later.

Use python 3.11 and SimPy 4.1.1

Patients are classified into 4 groups: Stroke, TIA, Complex neurological and other. Over the base period, the following number of patients were seen:
Acute stroke, n = 1320
TIA, n = 158
complex neurological, n = 456
other, n = 510 Please maintain these proportions in the simulation model.

New patients should arrive with the following mean inter-arrival times:
Acute stroke, 1.2 days
TIA, 9.3 days
Complex neurological, 3.6 days
other, 3.2 days 

For all groups, the time between arrival of new admissions should be modelled using an exponential probability distribution.

All time units are in days.

Inclusions:
1. Build and sample from distribution classes for arrivals (exponential) and for length of stay (lognormal) which will be used later so we can input specific seeds for reproducibility
2. Add a trace utility function as a debug print switch, set the TRACE constant to FALSE to turn tracing off
3. Create a parameter container class called Scenario that we can add to and change later
4. Create a patient class called AcutePatient that we can change as we proceed with the flow through the system later
5. Create a model class called StrokeUnit which will act as our simulation and start with the arrivals_generator for all our patient subtypes using the information above
6. Input and specify the model parameters as above
7. Leave comments for each main part of the inclusions above

In [2]:
import simpy
import numpy as np

# ---------------------------------------------------------
# 2. Add a trace utility function as a debug print switch
# ---------------------------------------------------------
TRACE = False

def trace(msg):
    """Utility function to print debug messages if TRACE is True."""
    if TRACE:
        print(msg)

# ---------------------------------------------------------
# 1. Build and sample from distribution classes
# ---------------------------------------------------------
class ExponentialDistribution:
    """Class to sample from an exponential distribution for arrivals."""
    def __init__(self, mean, random_seed=None):
        self.mean = mean
        # Using numpy's modern random generator for robust seeding
        self.rng = np.random.default_rng(seed=random_seed)
        
    def sample(self):
        # np.random.exponential takes the 'scale' parameter which is the mean
        return self.rng.exponential(scale=self.mean)

class LognormalDistribution:
    """Class to sample from a lognormal distribution for length of stay (for later use)."""
    def __init__(self, mu, sigma, random_seed=None):
        self.mu = mu
        self.sigma = sigma
        self.rng = np.random.default_rng(seed=random_seed)
        
    def sample(self):
        return self.rng.lognormal(mean=self.mu, sigma=self.sigma)

# ---------------------------------------------------------
# 3. Create a parameter container class called Scenario
# ---------------------------------------------------------
# 6. Input and specify the model parameters as above
# ---------------------------------------------------------
class Scenario:
    """Contains all parameters and configurations for the simulation."""
    def __init__(self):
        # Base period historical numbers (useful for validation or proportion calculations)
        self.base_patient_counts = {
            'Acute stroke': 1320,
            'TIA': 158,
            'Complex neurological': 456,
            'Other': 510
        }
        
        # Mean inter-arrival times (in days)
        # Note: Running independent exponential arrival streams using these means 
        # mathematically maintains the expected proportions of the base counts above.
        self.inter_arrival_means = {
            'Acute stroke': 1.2,
            'TIA': 9.3,
            'Complex neurological': 3.6,
            'Other': 3.2
        }
        
        # Base seed for reproducibility
        self.random_seed = 42

# ---------------------------------------------------------
# 4. Create a patient class called AcutePatient
# ---------------------------------------------------------
class AcutePatient:
    """Represents an individual patient flowing through the system."""
    def __init__(self, p_id, p_type, arrival_time):
        self.id = p_id
        self.type = p_type
        self.arrival_time = arrival_time
        
        # Attributes to be populated in later iterations
        self.length_of_stay = None
        self.discharge_time = None

# ---------------------------------------------------------
# 5. Create a model class called StrokeUnit
# ---------------------------------------------------------
class StrokeUnit:
    """The main simulation model encapsulating the environment and logic."""
    def __init__(self, env, scenario):
        self.env = env
        self.scenario = scenario
        self.patient_counter = 0
        
        # Initialize arrival distributions for each patient subtype
        self.arrival_distributions = {}
        for idx, (p_type, mean_iat) in enumerate(self.scenario.inter_arrival_means.items()):
            # Offset the seed for each subtype so their random streams are independent
            seed = self.scenario.random_seed + idx
            self.arrival_distributions[p_type] = ExponentialDistribution(mean=mean_iat, random_seed=seed)

    def arrivals_generator(self, patient_type):
        """Generates patient arrivals for a specific patient subtype."""
        while True:
            # Sample time until the next arrival for this specific type
            inter_arrival_time = self.arrival_distributions[patient_type].sample()
            
            # Wait for the arrival
            yield self.env.timeout(inter_arrival_time)
            
            # Create the new patient
            self.patient_counter += 1
            patient = AcutePatient(
                p_id=self.patient_counter,
                p_type=patient_type,
                arrival_time=self.env.now
            )
            
            # Log the arrival
            trace(f"Day {self.env.now:06.2f} | Patient ID {patient.id:04d} | Type: {patient.type} arrived.")
            
            # ---> Future iterations: Patient routing and resource requests will go here <---

    def run_simulation(self):
        """Starts the arrival generators for all patient subtypes."""
        # Start a separate generator process for each patient group
        for p_type in self.scenario.inter_arrival_means.keys():
            self.env.process(self.arrivals_generator(p_type))

# =========================================================
# Execution / Testing Block
# =========================================================
if __name__ == "__main__":
    # Temporarily set to True to verify the arrivals are working
    TRACE = True  
    
    print("--- Starting Acute and Community Stroke Service Simulation ---")
    
    # Initialize parameters
    scenario = Scenario()
    
    # Set up the SimPy environment
    env = simpy.Environment()
    
    # Instantiate and initialize the model
    model = StrokeUnit(env, scenario)
    model.run_simulation()
    
    # Run the simulation for a test period (e.g., 30 days)
    env.run(until=30)
    
    print(f"\nSimulation complete. Total patients arrived: {model.patient_counter}")

--- Starting Acute and Community Stroke Service Simulation ---
Day 001.33 | Patient ID 0001 | Type: Complex neurological arrived.
Day 001.54 | Patient ID 0002 | Type: TIA arrived.
Day 002.34 | Patient ID 0003 | Type: TIA arrived.
Day 002.44 | Patient ID 0004 | Type: Other arrived.
Day 002.84 | Patient ID 0005 | Type: TIA arrived.
Day 002.89 | Patient ID 0006 | Type: Acute stroke arrived.
Day 003.08 | Patient ID 0007 | Type: Complex neurological arrived.
Day 005.22 | Patient ID 0008 | Type: Other arrived.
Day 005.69 | Patient ID 0009 | Type: Acute stroke arrived.
Day 007.42 | Patient ID 0010 | Type: Complex neurological arrived.
Day 008.55 | Patient ID 0011 | Type: Acute stroke arrived.
Day 008.89 | Patient ID 0012 | Type: Acute stroke arrived.
Day 008.99 | Patient ID 0013 | Type: Acute stroke arrived.
Day 009.62 | Patient ID 0014 | Type: Other arrived.
Day 010.73 | Patient ID 0015 | Type: Other arrived.
Day 010.73 | Patient ID 0016 | Type: Acute stroke arrived.
Day 011.30 | Patient ID 